# Baseline Forecasting Models

This notebook implements classical forecasting baselines for hourly bike rental demand prediction.

The following models are evaluated:
- Naive Forecast,
- Seasonal Naive Forecast,
- AutoARIMA.

All models use the same chronological train-validation-test split. Validation and test evaluation are performed with the same 24-hour expanding-window approach used in the LightGBM and Chronos 2 notebooks.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from pmdarima import auto_arima

In [ ]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "processed" / "train.csv").exists():
            return candidate_path

    raise FileNotFoundError(
        "Could not find project root. Run this notebook from the repository "
        "or make sure data/processed/train.csv exists."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FORECAST_DIR = PROJECT_ROOT / "outputs" / "forecasts"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

os.makedirs(FORECAST_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

TARGET = "cnt"
FORECAST_HORIZON = 24
VALIDATION_STEP = FORECAST_HORIZON
VALIDATION_METRIC = "MASE"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load Chronological Data Splits

The train, validation and test sets were created previously during preprocessing. The train set is used as historical context, validation is used for model assessment, and test is used only for the final evaluation.

In [ ]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    index_col=0,
    parse_dates=True
)

val = pd.read_csv(
    DATA_DIR / "val.csv",
    index_col=0,
    parse_dates=True
)

test = pd.read_csv(
    DATA_DIR / "test.csv",
    index_col=0,
    parse_dates=True
)

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

In [ ]:
required_columns = [TARGET]

for split_name, split_df in {
    "train": train,
    "validation": val,
    "test": test
}.items():
    missing_columns = set(required_columns) - set(split_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in {split_name} split: {missing_columns}"
        )

In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "start": [train.index.min(), val.index.min(), test.index.min()],
    "end": [train.index.max(), val.index.max(), test.index.max()],
    "target_mean": [
        train[TARGET].mean(),
        val[TARGET].mean(),
        test[TARGET].mean()
    ]
})

split_summary

# Evaluation Metrics

The same metrics are used across baseline models: MAE, RMSE and MASE. MASE is calculated with a seasonal naive benchmark using a 24-hour seasonal period, which is appropriate for hourly data with daily seasonality.


In [ ]:
def rmse(
    y_true,
    y_pred
):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def mase(
    y_true,
    y_pred,
    train_series,
    seasonal_period=24
):
    seasonal_naive_error = np.mean(
        np.abs(
            train_series[seasonal_period:].values
            - train_series[:-seasonal_period].values
        )
    )

    if seasonal_naive_error == 0:
        raise ValueError(
            "MASE cannot be calculated because the seasonal naive forecast error is zero."
        )

    model_mae = mean_absolute_error(
        y_true,
        y_pred
    )

    return model_mae / seasonal_naive_error


def evaluate_forecasts(
    y_true,
    y_pred,
    train_series
):
    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "RMSE": rmse(
            y_true,
            y_pred
        ),
        "MASE": mase(
            y_true,
            y_pred,
            train_series,
            seasonal_period=FORECAST_HORIZON
        )
    }


# Expanding-Window Evaluation Logic

The evaluation follows the same idea as the LightGBM and Chronos 2 notebooks. For each window, the model receives all observations available up to the forecast origin and predicts the next 24 hours. Only complete 24-hour windows are evaluated.

In [ ]:
def get_forecast_starts(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    return range(
        0,
        len(evaluation_df) - forecast_horizon,
        step_size
    )


def summarize_expanding_window_setup(
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    forecast_starts = list(
        get_forecast_starts(
            evaluation_df,
            forecast_horizon,
            step_size
        )
    )

    evaluated_rows = len(forecast_starts) * forecast_horizon
    non_evaluated_rows = len(evaluation_df) - evaluated_rows

    return pd.DataFrame({
        "forecast_horizon": [forecast_horizon],
        "step_size": [step_size],
        "forecast_windows": [len(forecast_starts)],
        "available_rows": [len(evaluation_df)],
        "evaluated_rows": [evaluated_rows],
        "non_evaluated_rows": [non_evaluated_rows]
    })

In [ ]:
validation_window_summary = summarize_expanding_window_setup(
    val,
    FORECAST_HORIZON,
    VALIDATION_STEP
)

test_window_summary = summarize_expanding_window_setup(
    test,
    FORECAST_HORIZON
)

pd.concat(
    [
        validation_window_summary.assign(split="validation"),
        test_window_summary.assign(split="test")
    ],
    ignore_index=True
)[[
    "split",
    "forecast_horizon",
    "step_size",
    "forecast_windows",
    "available_rows",
    "evaluated_rows",
    "non_evaluated_rows"
]]

In [ ]:
def run_expanding_window_forecast(
    context_df,
    evaluation_df,
    forecast_function,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON
):
    all_predictions = []
    all_actuals = []
    all_timestamps = []
    all_window_starts = []

    forecast_starts = get_forecast_starts(
        evaluation_df,
        forecast_horizon,
        step_size
    )

    for start in forecast_starts:
        history = pd.concat([
            context_df,
            evaluation_df.iloc[:start]
        ])

        future = evaluation_df.iloc[
            start:start + forecast_horizon
        ]

        predictions = forecast_function(
            history,
            forecast_horizon
        )

        all_predictions.extend(
            predictions[:len(future)]
        )

        all_actuals.extend(
            future[TARGET].values
        )

        all_timestamps.extend(
            future.index
        )

        all_window_starts.extend(
            [future.index[0]] * len(future)
        )

    return pd.DataFrame({
        "timestamp": all_timestamps,
        "window_start": all_window_starts,
        "actual": all_actuals,
        "prediction": all_predictions
    })


def evaluate_expanding_window_results(
    forecast_df,
    train_series
):
    return evaluate_forecasts(
        forecast_df["actual"],
        forecast_df["prediction"],
        train_series
    )

# Naive Forecast

The Naive Forecast predicts every value in the next 24-hour horizon using the last observed value available at the forecast origin. This makes it a simple expanding-window multi-step baseline.

In [ ]:
def naive_forecast(
    history_df,
    forecast_horizon=FORECAST_HORIZON
):
    last_observation = history_df[TARGET].iloc[-1]

    return np.repeat(
        last_observation,
        forecast_horizon
    )

In [ ]:
naive_val_forecasts = run_expanding_window_forecast(
    context_df=train,
    evaluation_df=val,
    forecast_function=naive_forecast,
    step_size=VALIDATION_STEP
)

naive_val_metrics = evaluate_expanding_window_results(
    naive_val_forecasts,
    train[TARGET]
)

naive_val_metrics

# Seasonal Naive Forecast

The Seasonal Naive Forecast predicts the next 24-hour horizon using the last observed 24-hour pattern. This baseline directly represents daily seasonality in bike demand.

In [ ]:
def seasonal_naive_forecast(
    history_df,
    forecast_horizon=FORECAST_HORIZON,
    seasonal_period=24
):
    seasonal_pattern = history_df[TARGET].iloc[-seasonal_period:].values

    repetitions = int(
        np.ceil(forecast_horizon / seasonal_period)
    )

    return np.tile(
        seasonal_pattern,
        repetitions
    )[:forecast_horizon]

In [ ]:
seasonal_naive_val_forecasts = run_expanding_window_forecast(
    context_df=train,
    evaluation_df=val,
    forecast_function=seasonal_naive_forecast,
    step_size=VALIDATION_STEP
)

seasonal_naive_val_metrics = evaluate_expanding_window_results(
    seasonal_naive_val_forecasts,
    train[TARGET]
)

seasonal_naive_val_metrics

# AutoARIMA Forecast

AutoARIMA is used to select the best ARIMA parameters on the training set. The selected model is then evaluated with the same expanding-window logic, updating the fitted model with newly observed values before each forecast origin instead of re-running the full parameter search.

In [ ]:
ARIMA_UPDATE_MAXITER = 5


auto_arima_model = auto_arima(
    train[TARGET],
    seasonal=False,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
    error_action="ignore",
    max_p=3,
    max_q=3,
    max_d=2,
    n_jobs=1
)

AUTO_ARIMA_ORDER = auto_arima_model.order
AUTO_ARIMA_SEASONAL_ORDER = auto_arima_model.seasonal_order

print(auto_arima_model.summary())

In [ ]:
def run_auto_arima_expanding_forecast(
    fitted_model,
    evaluation_df,
    forecast_horizon=FORECAST_HORIZON,
    step_size=FORECAST_HORIZON,
    update_maxiter=ARIMA_UPDATE_MAXITER
):
    rolling_model = copy.deepcopy(
        fitted_model
    )

    all_predictions = []
    all_actuals = []
    all_timestamps = []
    all_window_starts = []
    observed_until = 0

    forecast_starts = get_forecast_starts(
        evaluation_df,
        forecast_horizon,
        step_size
    )

    for start in forecast_starts:
        new_observations = evaluation_df[TARGET].iloc[
            observed_until:start
        ]

        if len(new_observations) > 0:
            rolling_model.update(
                new_observations,
                maxiter=update_maxiter
            )

        future = evaluation_df.iloc[
            start:start + forecast_horizon
        ]

        predictions = rolling_model.predict(
            n_periods=forecast_horizon
        )

        all_predictions.extend(
            predictions[:len(future)]
        )

        all_actuals.extend(
            future[TARGET].values
        )

        all_timestamps.extend(
            future.index
        )

        all_window_starts.extend(
            [future.index[0]] * len(future)
        )

        observed_until = start

    return pd.DataFrame({
        "timestamp": all_timestamps,
        "window_start": all_window_starts,
        "actual": all_actuals,
        "prediction": all_predictions
    })

In [ ]:
auto_arima_val_forecasts = run_auto_arima_expanding_forecast(
    fitted_model=auto_arima_model,
    evaluation_df=val,
    step_size=VALIDATION_STEP
)

auto_arima_val_metrics = evaluate_expanding_window_results(
    auto_arima_val_forecasts,
    train[TARGET]
)

auto_arima_val_metrics

# Validation Comparison

The validation results are collected in one table. This makes the baseline notebook consistent with the model comparison approach used in the later notebooks.

In [ ]:
validation_results_df = pd.DataFrame([
    {
        "model": "naive",
        "model_details": "last observed value",
        **naive_val_metrics
    },
    {
        "model": "seasonal_naive",
        "model_details": "last observed 24-hour pattern",
        **seasonal_naive_val_metrics
    },
    {
        "model": "auto_arima",
        "model_details": f"order={AUTO_ARIMA_ORDER}, seasonal_order={AUTO_ARIMA_SEASONAL_ORDER}",
        **auto_arima_val_metrics
    }
])

validation_results_df = validation_results_df.sort_values(
    VALIDATION_METRIC
).reset_index(drop=True)

validation_results_df

# Final Test Evaluation

All baseline models are evaluated on the test set with the same 24-hour expanding-window approach. At this stage, the available history is `train + validation`, and the test set is used only for final model assessment.

In [ ]:
final_context = pd.concat([
    train,
    val
])

naive_test_forecasts = run_expanding_window_forecast(
    context_df=final_context,
    evaluation_df=test,
    forecast_function=naive_forecast,
    step_size=FORECAST_HORIZON
)

seasonal_naive_test_forecasts = run_expanding_window_forecast(
    context_df=final_context,
    evaluation_df=test,
    forecast_function=seasonal_naive_forecast,
    step_size=FORECAST_HORIZON
)

auto_arima_test_model = copy.deepcopy(
    auto_arima_model
)

auto_arima_test_model.update(
    val[TARGET],
    maxiter=ARIMA_UPDATE_MAXITER
)

auto_arima_test_forecasts = run_auto_arima_expanding_forecast(
    fitted_model=auto_arima_test_model,
    evaluation_df=test,
    step_size=FORECAST_HORIZON
)

In [ ]:
naive_test_metrics = evaluate_expanding_window_results(
    naive_test_forecasts,
    train[TARGET]
)

seasonal_naive_test_metrics = evaluate_expanding_window_results(
    seasonal_naive_test_forecasts,
    train[TARGET]
)

auto_arima_test_metrics = evaluate_expanding_window_results(
    auto_arima_test_forecasts,
    train[TARGET]
)

test_results_df = pd.DataFrame([
    {
        "model": "naive",
        "model_details": "last observed value",
        **naive_test_metrics
    },
    {
        "model": "seasonal_naive",
        "model_details": "last observed 24-hour pattern",
        **seasonal_naive_test_metrics
    },
    {
        "model": "auto_arima",
        "model_details": f"order={AUTO_ARIMA_ORDER}, seasonal_order={AUTO_ARIMA_SEASONAL_ORDER}",
        **auto_arima_test_metrics
    }
])

test_results_df = test_results_df.sort_values(
    VALIDATION_METRIC
).reset_index(drop=True)

test_results_df

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    naive_test_forecasts["actual"].values[:168],
    label="Actual"
)

plt.plot(
    naive_test_forecasts["prediction"].values[:168],
    label="Naive Forecast"
)

plt.plot(
    seasonal_naive_test_forecasts["prediction"].values[:168],
    label="Seasonal Naive Forecast"
)

plt.plot(
    auto_arima_test_forecasts["prediction"].values[:168],
    label="AutoARIMA Forecast"
)

plt.title("Baseline Test Forecasts (First 168 Hours)")
plt.xlabel("Forecast Horizon")
plt.ylabel("Bike Rentals")
plt.legend()

plt.show()

# Save Forecasts And Metrics

The final test predictions are saved in three separate CSV files, one per baseline model. Each forecast file contains only `actual` and `prediction`, matching the output format used by LightGBM and Chronos 2.

In [ ]:
naive_test_forecasts[["actual", "prediction"]].to_csv(
    FORECAST_DIR / "naive_test_forecasts.csv",
    index=False
)

seasonal_naive_test_forecasts[["actual", "prediction"]].to_csv(
    FORECAST_DIR / "seasonal_naive_test_forecasts.csv",
    index=False
)

auto_arima_test_forecasts[["actual", "prediction"]].to_csv(
    FORECAST_DIR / "auto_arima_test_forecasts.csv",
    index=False
)

validation_results_df.to_csv(
    METRICS_DIR / "baseline_validation_results.csv",
    index=False
)

test_results_df.to_csv(
    METRICS_DIR / "baseline_test_metrics.csv",
    index=False
)

test_results_df